In [2]:
## Travel package price prediction
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings 

warnings.filterwarnings("ignore")

%matplotlib inline

In [3]:
df=pd.read_csv("Travel.csv")
df.head()

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4888 entries, 0 to 4887
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CustomerID                4888 non-null   int64  
 1   ProdTaken                 4888 non-null   int64  
 2   Age                       4662 non-null   float64
 3   TypeofContact             4863 non-null   object 
 4   CityTier                  4888 non-null   int64  
 5   DurationOfPitch           4637 non-null   float64
 6   Occupation                4888 non-null   object 
 7   Gender                    4888 non-null   object 
 8   NumberOfPersonVisiting    4888 non-null   int64  
 9   NumberOfFollowups         4843 non-null   float64
 10  ProductPitched            4888 non-null   object 
 11  PreferredPropertyStar     4862 non-null   float64
 12  MaritalStatus             4888 non-null   object 
 13  NumberOfTrips             4748 non-null   float64
 14  Passport

### Data cleaning and handling 

In [5]:
### check all the categories
df['Gender'].value_counts()
df['MaritalStatus'].value_counts
df['TypeofContact'].value_counts

<bound method IndexOpsMixin.value_counts of 0          Self Enquiry
1       Company Invited
2          Self Enquiry
3       Company Invited
4          Self Enquiry
             ...       
4883       Self Enquiry
4884    Company Invited
4885       Self Enquiry
4886       Self Enquiry
4887       Self Enquiry
Name: TypeofContact, Length: 4888, dtype: object>

In [6]:
df['Gender'].value_counts()


Gender
Male       2916
Female     1817
Fe Male     155
Name: count, dtype: int64

In [7]:
df['MaritalStatus'].value_counts()

MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64

In [8]:
df['TypeofContact'].value_counts()

TypeofContact
Self Enquiry       3444
Company Invited    1419
Name: count, dtype: int64

In [9]:
df['Gender']=df['Gender'].replace('Fe Male','Female')
df['MaritalStatus']=df['MaritalStatus'].replace('Single','Unmarried')

In [10]:
df['Gender'].value_counts()


Gender
Male      2916
Female    1972
Name: count, dtype: int64

In [11]:
### checking ,missing values
## these are the values of the new fork
features_with_na=[features for features in df.columns if df[features].isnull().sum()>=1]
for feature in features_with_na:
    print(feature,np.round(df[feature].isnull().mean()*100,5), '%missing values')

Age 4.62357 %missing values
TypeofContact 0.51146 %missing values
DurationOfPitch 5.13502 %missing values
NumberOfFollowups 0.92062 %missing values
PreferredPropertyStar 0.53191 %missing values
NumberOfTrips 2.86416 %missing values
NumberOfChildrenVisiting 1.35025 %missing values
MonthlyIncome 4.76678 %missing values


In [12]:
## statistcs on numerical columns (Null Cols)
df[features_with_na].select_dtypes(exclude='object').describe()

,Age,DurationOfPitch,NumberOfFollowups,PreferredPropertyStar,NumberOfTrips,NumberOfChildrenVisiting,MonthlyIncome
count,4662.000000,4637.000000,4843.000000,4862.000000,4748.000000,4822.000000,4655.000000
mean,37.622265,15.490835,3.708445,3.581037,3.236521,1.187267,23619.853491
std,9.316387,8.519643,1.002509,0.798009,1.849019,0.857861,5380.698361
min,18.000000,5.000000,1.000000,3.000000,1.000000,0.000000,1000.000000
25%,31.000000,9.000000,3.000000,3.000000,2.000000,1.000000,20346.000000
50%,36.000000,13.000000,4.000000,3.000000,3.000000,1.000000,22347.000000
75%,44.000000,20.000000,4.000000,4.000000,4.000000,2.000000,25571.000000
max,61.000000,127.000000,6.000000,5.000000,22.000000,3.000000,98678.000000


### Imputing Null Values

In [13]:
## Age 
df.Age.fillna(df.Age.median(),inplace=True)

### Type of contact
df.TypeofContact.fillna(df.TypeofContact.mode([0]),inplace=True)

###DurationOfPitch
df.DurationOfPitch.fillna(df.DurationOfPitch.median(),inplace=True)

## NumberofFollowups
df.NumberOfFollowups.fillna(df.NumberOfFollowups.mode()[0],inplace=True)

### PrefferedPropertyStar
df.PreferredPropertyStar.fillna(df.PreferredPropertyStar.mode()[0],inplace=True)

#NumberofTrips
df.NumberOfTrips.fillna(df.NumberOfTrips.median,inplace=True)

##NumberOfChildrenVisting
df.NumberOfChildrenVisiting.fillna(df.NumberOfChildrenVisiting.mode()[0],inplace=True)

##Montly Income 
df.MonthlyIncome.fillna(df.MonthlyIncome.median(),inplace=True)

In [14]:
df.head()

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Unmarried,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,36.0,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [15]:
df.drop('CustomerID',inplace=True,axis=1)

### Feature Enginerring

#### Feature Extraction


In [16]:
df['TotalVisting']=df['NumberOfPersonVisiting']+df['NumberOfChildrenVisiting']
df.drop(columns=['NumberOfPersonVisiting','NumberOfChildrenVisiting'],axis=1,inplace=True)

In [17]:
### getting the numerical features 
num_features=[feature for feature in df.columns if df[feature].dtype !='O']
print('Num of Numerical features:',len(num_features))

Num of Numerical features: 11


In [18]:
### getting the categorical features 
cat_features=[feature for feature in df.columns if df[feature].dtype=='O']
print('Num of the categorical features is:',len(cat_features))

Num of the categorical features is: 7


In [19]:
## Discrete features
discrete_features=[feature for feature in num_features if len(df[feature]<=25)]
print('discrete features for the given dataset is :',len(discrete_features))

discrete features for the given dataset is : 11


In [20]:
### continuous features 
continuous_features=[feature for feature in num_features if feature not in discrete_features]
print('num of continuous features:',len(continuous_features))

num of continuous features: 0


In [21]:
df.head()

,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalVisting
0,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,Manager,20993.0,3.0
1,0,49.0,Company Invited,1,14.0,Salaried,Male,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,Manager,20130.0,5.0
2,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,4.0,Basic,3.0,Unmarried,7.0,1,3,0,Executive,17090.0,3.0
3,0,33.0,Company Invited,1,9.0,Salaried,Female,3.0,Basic,3.0,Divorced,2.0,1,5,1,Executive,17909.0,3.0
4,0,36.0,Self Enquiry,1,8.0,Small Business,Male,3.0,Basic,4.0,Divorced,1.0,0,5,1,Executive,18468.0,2.0


### Train Test And Model Training


In [22]:
from sklearn.model_selection import train_test_split
x=df.drop(['ProdTaken'],axis=1)
y=df['ProdTaken']

In [23]:
x.head()

,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalVisting
0,41.0,Self Enquiry,3,6.0,Salaried,Female,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,Manager,20993.0,3.0
1,49.0,Company Invited,1,14.0,Salaried,Male,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,Manager,20130.0,5.0
2,37.0,Self Enquiry,1,8.0,Free Lancer,Male,4.0,Basic,3.0,Unmarried,7.0,1,3,0,Executive,17090.0,3.0
3,33.0,Company Invited,1,9.0,Salaried,Female,3.0,Basic,3.0,Divorced,2.0,1,5,1,Executive,17909.0,3.0
4,36.0,Self Enquiry,1,8.0,Small Business,Male,3.0,Basic,4.0,Divorced,1.0,0,5,1,Executive,18468.0,2.0


In [24]:
y.value_counts()

ProdTaken
0    3968
1     920
Name: count, dtype: int64

In [25]:
x.head()


,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalVisting
0,41.0,Self Enquiry,3,6.0,Salaried,Female,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,Manager,20993.0,3.0
1,49.0,Company Invited,1,14.0,Salaried,Male,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,Manager,20130.0,5.0
2,37.0,Self Enquiry,1,8.0,Free Lancer,Male,4.0,Basic,3.0,Unmarried,7.0,1,3,0,Executive,17090.0,3.0
3,33.0,Company Invited,1,9.0,Salaried,Female,3.0,Basic,3.0,Divorced,2.0,1,5,1,Executive,17909.0,3.0
4,36.0,Self Enquiry,1,8.0,Small Business,Male,3.0,Basic,4.0,Divorced,1.0,0,5,1,Executive,18468.0,2.0


In [26]:
## train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=40)
x_train.shape,x_test.shape

((3910, 17), (978, 17))

In [27]:
## creating the column Transformer with 3types of trandformers
cat_features=x.select_dtypes(include="object").columns
num_features=x.select_dtypes(exclude="object").columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer=StandardScaler()
oh_trasformer=OneHotEncoder(drop='first',handle_unknown='ignore', sparse_output=False)

preprocessor=ColumnTransformer(
    [
        ("OneHotEncoder",oh_trasformer,cat_features),
        ("Standardscaler",numeric_transformer,num_features)
    ]
)

In [28]:
preprocessor

,transformers,"[('OneHotEncoder', ...), ('Standardscaler', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,'first'
,sparse_output,False


In [29]:
x_train

,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalVisting
2753,32.0,Self Enquiry,1,12.0,Large Business,Male,4.0,Basic,3.0,Divorced,2.0,1,4,0,Executive,23499.0,5.0
1262,43.0,Company Invited,1,23.0,Large Business,Male,4.0,Basic,3.0,Married,1.0,1,5,1,Executive,17437.0,3.0
542,38.0,Self Enquiry,1,6.0,Salaried,Female,3.0,Standard,3.0,Unmarried,5.0,1,2,0,Senior Manager,22861.0,3.0
3509,56.0,Self Enquiry,1,11.0,Salaried,Male,4.0,Deluxe,3.0,Divorced,2.0,1,4,0,Manager,22713.0,3.0
1056,28.0,Company Invited,3,6.0,Large Business,Male,3.0,Basic,3.0,Divorced,4.0,0,3,1,Executive,17909.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3603,40.0,Company Invited,1,11.0,Small Business,Female,4.0,Deluxe,3.0,Unmarried,2.0,0,3,1,Manager,23720.0,4.0
4722,30.0,Self Enquiry,1,35.0,Salaried,Female,4.0,Basic,3.0,Married,6.0,0,5,1,Executive,21192.0,5.0
3340,32.0,Self Enquiry,1,31.0,Small Business,Female,5.0,Deluxe,5.0,Unmarried,3.0,0,5,0,Manager,25490.0,7.0
3064,36.0,Self Enquiry,1,34.0,Small Business,Female,5.0,Basic,5.0,Unmarried,3.0,0,4,1,Executive,21237.0,5.0


In [30]:
cat_features

Index(['TypeofContact', 'Occupation', 'Gender', 'ProductPitched',
       'MaritalStatus', 'NumberOfTrips', 'Designation'],
      dtype='object')

In [31]:
num_features

Index(['Age', 'CityTier', 'DurationOfPitch', 'NumberOfFollowups',
       'PreferredPropertyStar', 'Passport', 'PitchSatisfactionScore', 'OwnCar',
       'MonthlyIncome', 'TotalVisting'],
      dtype='object')

In [32]:
x_train

,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalVisting
2753,32.0,Self Enquiry,1,12.0,Large Business,Male,4.0,Basic,3.0,Divorced,2.0,1,4,0,Executive,23499.0,5.0
1262,43.0,Company Invited,1,23.0,Large Business,Male,4.0,Basic,3.0,Married,1.0,1,5,1,Executive,17437.0,3.0
542,38.0,Self Enquiry,1,6.0,Salaried,Female,3.0,Standard,3.0,Unmarried,5.0,1,2,0,Senior Manager,22861.0,3.0
3509,56.0,Self Enquiry,1,11.0,Salaried,Male,4.0,Deluxe,3.0,Divorced,2.0,1,4,0,Manager,22713.0,3.0
1056,28.0,Company Invited,3,6.0,Large Business,Male,3.0,Basic,3.0,Divorced,4.0,0,3,1,Executive,17909.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3603,40.0,Company Invited,1,11.0,Small Business,Female,4.0,Deluxe,3.0,Unmarried,2.0,0,3,1,Manager,23720.0,4.0
4722,30.0,Self Enquiry,1,35.0,Salaried,Female,4.0,Basic,3.0,Married,6.0,0,5,1,Executive,21192.0,5.0
3340,32.0,Self Enquiry,1,31.0,Small Business,Female,5.0,Deluxe,5.0,Unmarried,3.0,0,5,0,Manager,25490.0,7.0
3064,36.0,Self Enquiry,1,34.0,Small Business,Female,5.0,Basic,5.0,Unmarried,3.0,0,4,1,Executive,21237.0,5.0


In [33]:
pd.DataFrame(x_train)

,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalVisting
2753,32.0,Self Enquiry,1,12.0,Large Business,Male,4.0,Basic,3.0,Divorced,2.0,1,4,0,Executive,23499.0,5.0
1262,43.0,Company Invited,1,23.0,Large Business,Male,4.0,Basic,3.0,Married,1.0,1,5,1,Executive,17437.0,3.0
542,38.0,Self Enquiry,1,6.0,Salaried,Female,3.0,Standard,3.0,Unmarried,5.0,1,2,0,Senior Manager,22861.0,3.0
3509,56.0,Self Enquiry,1,11.0,Salaried,Male,4.0,Deluxe,3.0,Divorced,2.0,1,4,0,Manager,22713.0,3.0
1056,28.0,Company Invited,3,6.0,Large Business,Male,3.0,Basic,3.0,Divorced,4.0,0,3,1,Executive,17909.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3603,40.0,Company Invited,1,11.0,Small Business,Female,4.0,Deluxe,3.0,Unmarried,2.0,0,3,1,Manager,23720.0,4.0
4722,30.0,Self Enquiry,1,35.0,Salaried,Female,4.0,Basic,3.0,Married,6.0,0,5,1,Executive,21192.0,5.0
3340,32.0,Self Enquiry,1,31.0,Small Business,Female,5.0,Deluxe,5.0,Unmarried,3.0,0,5,0,Manager,25490.0,7.0
3064,36.0,Self Enquiry,1,34.0,Small Business,Female,5.0,Basic,5.0,Unmarried,3.0,0,4,1,Executive,21237.0,5.0


In [34]:
x_test

,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalVisting
141,36.0,Self Enquiry,1,35.0,Small Business,Male,3.0,Basic,3.0,Unmarried,6.0,0,2,1,Executive,18452.0,2.0
3894,36.0,Self Enquiry,2,33.0,Salaried,Female,4.0,Standard,3.0,Unmarried,3.0,1,1,1,Senior Manager,27515.0,4.0
1663,43.0,Company Invited,1,13.0,Small Business,Male,1.0,Basic,3.0,Married,5.0,0,4,1,Executive,17089.0,2.0
2098,37.0,Self Enquiry,3,22.0,Small Business,Male,4.0,Deluxe,3.0,Married,5.0,0,5,1,Manager,21334.0,3.0
4733,29.0,Self Enquiry,1,9.0,Salaried,Male,5.0,Basic,4.0,Married,2.0,0,1,0,Executive,21879.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2222,36.0,Self Enquiry,1,22.0,Salaried,Female,1.0,Basic,5.0,Unmarried,2.0,0,1,1,Executive,17743.0,2.0
4692,47.0,Self Enquiry,1,15.0,Salaried,Female,5.0,Deluxe,5.0,Married,2.0,1,1,1,Manager,23293.0,7.0
4006,31.0,Self Enquiry,3,11.0,Small Business,Female,5.0,Deluxe,4.0,Married,3.0,1,3,1,Manager,23887.0,5.0
1889,30.0,Company Invited,1,9.0,Salaried,Male,3.0,Basic,3.0,Married,5.0,0,3,1,Executive,17097.0,2.0


In [35]:
x_test

,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalVisting
141,36.0,Self Enquiry,1,35.0,Small Business,Male,3.0,Basic,3.0,Unmarried,6.0,0,2,1,Executive,18452.0,2.0
3894,36.0,Self Enquiry,2,33.0,Salaried,Female,4.0,Standard,3.0,Unmarried,3.0,1,1,1,Senior Manager,27515.0,4.0
1663,43.0,Company Invited,1,13.0,Small Business,Male,1.0,Basic,3.0,Married,5.0,0,4,1,Executive,17089.0,2.0
2098,37.0,Self Enquiry,3,22.0,Small Business,Male,4.0,Deluxe,3.0,Married,5.0,0,5,1,Manager,21334.0,3.0
4733,29.0,Self Enquiry,1,9.0,Salaried,Male,5.0,Basic,4.0,Married,2.0,0,1,0,Executive,21879.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2222,36.0,Self Enquiry,1,22.0,Salaried,Female,1.0,Basic,5.0,Unmarried,2.0,0,1,1,Executive,17743.0,2.0
4692,47.0,Self Enquiry,1,15.0,Salaried,Female,5.0,Deluxe,5.0,Married,2.0,1,1,1,Manager,23293.0,7.0
4006,31.0,Self Enquiry,3,11.0,Small Business,Female,5.0,Deluxe,4.0,Married,3.0,1,3,1,Manager,23887.0,5.0
1889,30.0,Company Invited,1,9.0,Salaried,Male,3.0,Basic,3.0,Married,5.0,0,3,1,Executive,17097.0,2.0


In [36]:
x_test

,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalVisting
141,36.0,Self Enquiry,1,35.0,Small Business,Male,3.0,Basic,3.0,Unmarried,6.0,0,2,1,Executive,18452.0,2.0
3894,36.0,Self Enquiry,2,33.0,Salaried,Female,4.0,Standard,3.0,Unmarried,3.0,1,1,1,Senior Manager,27515.0,4.0
1663,43.0,Company Invited,1,13.0,Small Business,Male,1.0,Basic,3.0,Married,5.0,0,4,1,Executive,17089.0,2.0
2098,37.0,Self Enquiry,3,22.0,Small Business,Male,4.0,Deluxe,3.0,Married,5.0,0,5,1,Manager,21334.0,3.0
4733,29.0,Self Enquiry,1,9.0,Salaried,Male,5.0,Basic,4.0,Married,2.0,0,1,0,Executive,21879.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2222,36.0,Self Enquiry,1,22.0,Salaried,Female,1.0,Basic,5.0,Unmarried,2.0,0,1,1,Executive,17743.0,2.0
4692,47.0,Self Enquiry,1,15.0,Salaried,Female,5.0,Deluxe,5.0,Married,2.0,1,1,1,Manager,23293.0,7.0
4006,31.0,Self Enquiry,3,11.0,Small Business,Female,5.0,Deluxe,4.0,Married,3.0,1,3,1,Manager,23887.0,5.0
1889,30.0,Company Invited,1,9.0,Salaried,Male,3.0,Basic,3.0,Married,5.0,0,3,1,Executive,17097.0,2.0


In [38]:
x_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3910 entries, 2753 to 3398
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Age                     3910 non-null   float64
 1   TypeofContact           3888 non-null   object 
 2   CityTier                3910 non-null   int64  
 3   DurationOfPitch         3910 non-null   float64
 4   Occupation              3910 non-null   object 
 5   Gender                  3910 non-null   object 
 6   NumberOfFollowups       3910 non-null   float64
 7   ProductPitched          3910 non-null   object 
 8   PreferredPropertyStar   3910 non-null   float64
 9   MaritalStatus           3910 non-null   object 
 10  NumberOfTrips           3910 non-null   object 
 11  Passport                3910 non-null   int64  
 12  PitchSatisfactionScore  3910 non-null   int64  
 13  OwnCar                  3910 non-null   int64  
 14  Designation             3910 non-null   ob

In [ ]:
x_train

,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalVisting
2753,32.0,Self Enquiry,1,12.0,Large Business,Male,4.0,Basic,3.0,Divorced,2.0,1,4,0,Executive,23499.0,5.0
1262,43.0,Company Invited,1,23.0,Large Business,Male,4.0,Basic,3.0,Married,1.0,1,5,1,Executive,17437.0,3.0
542,38.0,Self Enquiry,1,6.0,Salaried,Female,3.0,Standard,3.0,Unmarried,5.0,1,2,0,Senior Manager,22861.0,3.0
3509,56.0,Self Enquiry,1,11.0,Salaried,Male,4.0,Deluxe,3.0,Divorced,2.0,1,4,0,Manager,22713.0,3.0
1056,28.0,Company Invited,3,6.0,Large Business,Male,3.0,Basic,3.0,Divorced,4.0,0,3,1,Executive,17909.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3603,40.0,Company Invited,1,11.0,Small Business,Female,4.0,Deluxe,3.0,Unmarried,2.0,0,3,1,Manager,23720.0,4.0
4722,30.0,Self Enquiry,1,35.0,Salaried,Female,4.0,Basic,3.0,Married,6.0,0,5,1,Executive,21192.0,5.0
3340,32.0,Self Enquiry,1,31.0,Small Business,Female,5.0,Deluxe,5.0,Unmarried,3.0,0,5,0,Manager,25490.0,7.0
3064,36.0,Self Enquiry,1,34.0,Small Business,Female,5.0,Basic,5.0,Unmarried,3.0,0,4,1,Executive,21237.0,5.0


In [42]:
for col in x_train.columns:
    print(col, x_train[col].apply(type).unique())

Age [<class 'float'>]
TypeofContact [<class 'str'> <class 'float'>]
CityTier [<class 'int'>]
DurationOfPitch [<class 'float'>]
Occupation [<class 'str'>]
Gender [<class 'str'>]
NumberOfFollowups [<class 'float'>]
ProductPitched [<class 'str'>]
PreferredPropertyStar [<class 'float'>]
MaritalStatus [<class 'str'>]
NumberOfTrips [<class 'float'> <class 'method'>]
Passport [<class 'int'>]
PitchSatisfactionScore [<class 'int'>]
OwnCar [<class 'int'>]
Designation [<class 'str'>]
MonthlyIncome [<class 'float'>]
TotalVisting [<class 'float'>]


In [45]:
for col in x_train.select_dtypes(include='object').columns:
    x_train[col] = x_train[col].astype(str)
    x_test[col] = x_test[col].astype(str)

In [46]:
for col in x_train.columns:
    if x_train[col].dtype == 'object':
        x_train[col].fillna('missing', inplace=True)
        x_test[col].fillna('missing', inplace=True)
    else:
        x_train[col].fillna(x_train[col].mean(), inplace=True)
        x_test[col].fillna(x_test[col].mean(), inplace=True)

In [47]:
x_train=preprocessor.fit_transform(x_train)


In [48]:
x_test=preprocessor.fit_transform(x_test)

In [49]:
print(x_train.shape)
print(x_test.shape)

(3910, 36)
(978, 36)


### Model training

In [50]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report,ConfusionMatrixDisplay,\
precision_score,recall_score,f1_score,roc_auc_score,roc_curve

In [52]:
models={
    "Random Forest":RandomForestClassifier(),
    "Logistic Regression":LogisticRegression(),
    "Gradient Boosting":GradientBoostingClassifier(),
    "Adaboost":AdaBoostClassifier()
    
}
for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(x_train,y_train)## train model
    
    ## make predictions
    y_train_pred=model.predict(x_train)
    y_test_pred=model.predict(x_test)
    
    ### trarining set performance 
    model_train_accuracy=accuracy_score(y_train,y_train_pred)
    model_train_f1=f1_score(y_train,y_train_pred,average="weighted")
    model_train_precision=precision_score(y_train,y_train_pred)
    model_train_recall=recall_score(y_train,y_train_pred)
    model_train_rocauc_score=roc_auc_score(y_train,y_train_pred)
    
    ## test performance
    model_test_accuracy=accuracy_score(y_test,y_test_pred)
    model_test_f1=f1_score(y_test,y_test_pred,average="weighted")
    model_test_precision=precision_score(y_test,y_test_pred)
    model_test_recall=recall_score(y_test,y_test_pred)
    model_test_rocauc_score=roc_auc_score(y_test,y_test_pred)
    
    print(list(models.keys())[i])
    
    print("model performance for the training set  ")
    print("-Accuracy: {:.4f}".format(model_train_accuracy))
    print("- F1 score:{:.4f}".format(model_train_precision))
    print(" -Precision:{:.4f}".format(model_train_precision))
    print('-recall: {:.4f}' .format(model_train_recall))
    print('-roc auc score: {:.4f}'.format(model_train_rocauc_score)) 
    
      
       
    print("model performance of the test datasset ")   
   
    print("-Accuracy: {:.4f}".format(model_test_accuracy))
    print("- F1 score:{:.4f}".format(model_test_precision))
    print(" -Precision:{:.4f}".format(model_test_precision))
    print('-recall: {:.4f}' .format(model_test_recall))
    print('-roc auc score: {:.4f}'.format(model_test_rocauc_score)) 
    
       
    

Random Forest
model performance for the training set  
-Accuracy: 1.0000
- F1 score:1.0000
 -Precision:1.0000
-recall: 1.0000
-roc auc score: 1.0000
model performance of the test datasset 
-Accuracy: 0.9182
- F1 score:0.9237
 -Precision:0.9237
-recall: 0.6056
-roc auc score: 0.7971
Logistic Regression
model performance for the training set  
-Accuracy: 0.8422
- F1 score:0.6762
 -Precision:0.6762
-recall: 0.3189
-roc auc score: 0.6416
model performance of the test datasset 
-Accuracy: 0.8548
- F1 score:0.7436
 -Precision:0.7436
-recall: 0.3222
-roc auc score: 0.6486
Gradient Boosting
model performance for the training set  
-Accuracy: 0.8882
- F1 score:0.8633
 -Precision:0.8633
-recall: 0.4865
-roc auc score: 0.7343
model performance of the test datasset 
-Accuracy: 0.8630
- F1 score:0.7396
 -Precision:0.7396
-recall: 0.3944
-roc auc score: 0.6816
Adaboost
model performance for the training set  
-Accuracy: 0.8448
- F1 score:0.7737
 -Precision:0.7737
-recall: 0.2541
-roc auc score: 0.61

In [53]:
## Hyperparameter Training 
rf_params={"max_depth":[5,8,15,None,10],
           "max_features":[5,7,"auto",],
           "min_samples_split":[2,8,15,20],
           "n_estimators":[100,200,300,500,1000]}
adaboost_params={
    "n_estimators":[50,60,70,80,90],
    "algorithm":['SAMME','SAMME.R']
    
}

In [54]:
## Models list for Hyperparammeter tuning
randomcv_models=[
    ("RF",RandomForestClassifier(),rf_params),
    ("AB",AdaBoostClassifier(),adaboost_params)
]

In [55]:
from sklearn.model_selection import RandomizedSearchCV

model_param={}
for name,model,params in randomcv_models:
    random=RandomizedSearchCV(estimator=model,
                              param_distributions=params,
                              n_iter=100,
                              cv=3,
                              verbose=2,
                              n_jobs=-1)
    random.fit(x_train,y_train)
    random.fit(x_train,y_train)
    model_param[name]=random.best_params_
    
    
    for model_name in model_param:
        print(f"----------------------Best params for {model_name}")
        print(model_param[model_name])

Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
----------------------Best params for RF
{'n_estimators': 300, 'min_samples_split': 2, 'max_features': 7, 'max_depth': None}
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Fitting 3 folds for each of 10 candidates, totalling 30 fits
----------------------Best params for RF
{'n_estimators': 300, 'min_samples_split': 2, 'max_features': 7, 'max_depth': None}
----------------------Best params for AB
{'n_estimators': 80, 'algorithm': 'SAMME'}


In [56]:
models={
    "Random Forest":RandomForestClassifier(n_estimators=300,min_samples_split=2,max_features=7,max_depth=None),
    "AB":AdaBoostClassifier(n_estimators=80,algorithm='SAMME')
    
}
for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(x_train,y_train)## train model
    
    ## make predictions
    y_train_pred=model.predict(x_train)
    y_test_pred=model.predict(x_test)
    
    ### trarining set performance 
    model_train_accuracy=accuracy_score(y_train,y_train_pred)
    model_train_f1=f1_score(y_train,y_train_pred,average="weighted")
    model_train_precision=precision_score(y_train,y_train_pred)
    model_train_recall=recall_score(y_train,y_train_pred)
    model_train_rocauc_score=roc_auc_score(y_train,y_train_pred)
    
    ## test performance
    model_test_accuracy=accuracy_score(y_test,y_test_pred)
    model_test_f1=f1_score(y_test,y_test_pred,average="weighted")
    model_test_precision=precision_score(y_test,y_test_pred)
    model_test_recall=recall_score(y_test,y_test_pred)
    model_test_rocauc_score=roc_auc_score(y_test,y_test_pred)
    
    print(list(models.keys())[i])
    
    print("model performance for the training set  ")
    print("-Accuracy: {:.4f}".format(model_train_accuracy))
    print("- F1 score:{:.4f}".format(model_train_precision))
    print(" -Precision:{:.4f}".format(model_train_precision))
    print('-recall: {:.4f}' .format(model_train_recall))
    print('-roc auc score: {:.4f}'.format(model_train_rocauc_score)) 
    
      
       
    print("model performance of the test datasset ")   
    print("model performance for the training set  ")
    print("-Accuracy: {:.4f}".format(model_test_accuracy))
    print("- F1 score:{:.4f}".format(model_test_precision))
    print(" -Precision:{:.4f}".format(model_test_precision))
    print('-recall: {:.4f}' .format(model_test_recall))
    print('-roc auc score: {:.4f}'.format(model_test_rocauc_score)) 
    
       
    

Random Forest
model performance for the training set  
-Accuracy: 1.0000
- F1 score:1.0000
 -Precision:1.0000
-recall: 1.0000
-roc auc score: 1.0000
model performance of the test datasset 
model performance for the training set  
-Accuracy: 0.9233
- F1 score:0.9268
 -Precision:0.9268
-recall: 0.6333
-roc auc score: 0.8110
AB
model performance for the training set  
-Accuracy: 0.8430
- F1 score:0.7669
 -Precision:0.7669
-recall: 0.2446
-roc auc score: 0.6136
model performance of the test datasset 
model performance for the training set  
-Accuracy: 0.8466
- F1 score:0.7500
 -Precision:0.7500
-recall: 0.2500
-roc auc score: 0.6156
